# Final Prediction
This file tends to combine our prediction / results from Model 1 (Basic Statistics Model) with Model 2 (Sentiment / Popularity Analsysis). The combination final score calculation method is shown below in the Preview section.
- Training Season (20, 21, 22, 23)
- Test Season (24)

**Preview**:
- A = ALPHA, B = BETA
- ALPHA / BETA are determined through grid search (maximizing precison and recall)
- Formula = A * (p_ensemble) + (1 - A) * [B * (espn_sentiment_score) + (1 - B) * (popularity_score)]



## Imports (Library / Files / Actual)

In [ ]:
import pandas as pd
import numpy as np



> Import Training Prediction



In [ ]:
# Model 1
model_1_train = pd.read_csv('2020_to_2023_All_Star_Predictions.csv')
model_1_train = model_1_train.rename(columns={"Year": "season"})
# adjust columns
model_1_train["season"] = model_1_train["season"].replace({
    2020: "2019-2020",
    2021: "2020-2021",
    2022: "2021-2022",
    2023: "2022-2023",
    2024: "2023-2024"
})

# Model 2
model_2_train_popularity = pd.read_csv('google_trends_training_df.csv')
model_2_train_popularity = model_2_train_popularity.rename(columns={"player": "Player"})
model_2_train_espn_score = pd.read_csv('espn_final_scores_train.csv')



> Import Testing Prediction



In [ ]:
# Model 1
model_1_test = pd.read_csv('2024_All_Star_Predictions.csv')

# Model 2
model_2_test_popularity = pd.read_csv('google_trends_test_df.csv')
model_2_test_espn_score = pd.read_csv('espn_final_scores_test_23_24.csv')

# fix "player" to "Player"
model_2_test_popularity = model_2_test_popularity.rename(columns={"player": "Player"})



> Import Current Season (25/26) Prediction



In [ ]:
# Model 1
model_1_current = pd.read_csv('2026_All_Star_Predictions.csv')

# Model 2
model_2_current_popularity = pd.read_csv('google_trends_pred_25_26.csv')



> Import Actual All Star



In [ ]:
all_stars_23_24 = ['Giannis Antetokounmpo', 'Bam Adebayo', 'Tyrese Haliburton', 'Damian Lillard', 'Jayson Tatum', 'Paolo Banchero',
                'Jaylen Brown', 'Jalen Brunson', 'Tyrese Maxey', 'Donovan Mitchell', 'Julius Randle', 'Joel Embiid',
                'LeBron James', 'Luka Dončić', 'Kevin Durant', 'Shai Gilgeous-Alexander', 'Nikola Jokić', 'Devin Booker', 'Stephen Curry', 'Anthony Davis',
                'Anthony Edwards', 'Paul George', 'Kawhi Leonard', 'Karl-Anthony Towns']


all_stars_22_23 = ['Giannis Antetokounmpo', 'Lauri Markkanen', 'Donovan Mitchell', 'Ja Morant', 'Jayson Tatum', 'Bam Adebayo',
                'Shai Gilgeous-Alexander', 'DeMar DeRozan', 'Jrue Holiday', 'Damian Lillard', 'Domantas Sabonis', 'Kevin Durant',
                'LeBron James', 'Luka Dončić', 'Joel Embiid', 'Kyrie Irving', 'Nikola Jokić', 'Jaylen Brown', 'Stephen Curry', 'Paul George',
                'Tyrese Haliburton', 'Jaren Jackson Jr.', 'Julius Randle', 'Zion Williamson']


all_stars_21_22 = ['Kevin Durant', 'Joel Embiid', 'Ja Morant', 'Jayson Tatum', 'Andrew Wiggins', 'Trae Young',
                'Devin Booker', 'Rudy Gobert', 'Zach LaVine', 'Khris Middleton', 'Karl-Anthony Towns', 'Draymond Green',
                'LeBron James', 'Luka Dončić', 'Giannis Antetokounmpo', 'Stephen Curry', 'Nikola Jokić', 'DeMar DeRozan', 'Jimmy Butler', 'Darius Garland',
                'James Harden', 'Donovan Mitchell', 'Chris Paul', 'Fred VanVleet']


all_stars_20_21 = ['Giannis Antetokounmpo', 'Stephen Curry', 'Luka Dončić', 'LeBron James', 'Nikola Jokić', 'Jaylen Brown',
                'Paul George', 'Rudy Gobert', 'Damian Lillard', 'Chris Paul', 'Ben Simmons',
                'Bradley Beal', 'Kyrie Irving', 'Kawhi Leonard', 'Jayson Tatum', 'Zion Williamson', 'James Harden', 'Zach LaVine', 'Donovan Mitchell',
                'Julius Randle', 'Nikola Vučević', 'Kevin Durant', 'Anthony Davis', 'Joel Embiid']


all_stars_19_20 = ['Anthony Davis', 'Kawhi Leonard', 'Luka Dončić', 'Damian Lillard', 'James Harden', 'Ben Simmons',
                'Nikola Jokić', 'Jayson Tatum', 'Chris Paul', 'Russell Westbrook', 'Domantas Sabonis', 'LeBron James',
                'Giannis Antetokounmpo', 'Joel Embiid', 'Pascal Siakam', 'Kemba Walker', 'Trae Young', 'Khris Middleton', 'Bam Adebayo', 'Rudy Gobert',
                'Jimmy Butler', 'Kyle Lowry', 'Brandon Ingram', 'Donovan Mitchell']

## Combine Results

> A = ALPHA, B = BETA

> ALPHA / BETA are determined through grid search (maximizing precison and recall)

> Formula = A * (p_ensemble) + (1 - A) * [B * (espn_sentiment_score) + (1 - B) * (popularity_score)]



In [ ]:
df_train_final = (
    model_1_train[["Player", "season", "p_ensemble"]]
    .merge(
        model_2_train_popularity[["Player", "season", "final_score"]]
        .rename(columns={"final_score": "popularity_score"}),
        on=["Player", "season"],
        how="inner"
    )
    .merge(
        model_2_train_espn_score[["Player", "season", "final_sentiment_score"]]
        .rename(columns={"final_sentiment_score": "espn_score"}),
        on=["Player", "season"],
        how="inner"
    )
)

print(df_train_final.head())
print(df_train_final.shape)

         Player     season  p_ensemble  popularity_score  espn_score
0   Luka Dončić  2022-2023    0.964862          0.070382    0.717518
1  LeBron James  2021-2022    0.963797          0.307217    1.000000
2  James Harden  2019-2020    0.956675          0.166341    0.764383
3  Nikola Jokić  2021-2022    0.955922          0.055166    0.577263
4   Luka Dončić  2021-2022    0.954237          0.055292    0.667190
(1540, 5)


In [ ]:
# Combine Table / DataFrame
df_test_final = (
    model_1_test[["Player", "p_ensemble"]]
    .merge(
        model_2_test_popularity[["Player", "final_score"]]
        .rename(columns={"final_score": "popularity_score"}),
        on="Player",
        how="inner"
    )
    .merge(
        model_2_test_espn_score[["Player", "final_sentiment_score"]]
        .rename(columns={"final_sentiment_score": "espn_score"}),
        on="Player",
        how="inner"
    )
)

print(df_test_final.head())
print(df_test_final.shape)

                    Player  p_ensemble  popularity_score  espn_score
0              Luka Dončić    0.954940          0.059965    0.718753
1             Nikola Jokić    0.940473          0.058163    0.714385
2    Giannis Antetokounmpo    0.930771          0.096008    0.731526
3  Shai Gilgeous-Alexander    0.920671          0.081206    0.647467
4             Jayson Tatum    0.905790          0.087142    0.633396
(429, 4)


# Evaluate Final Training Predictions



> Precision / Recall Function


> Compute Final Score Method





In [ ]:
def precision_recall_at_k(df, k, label_col="is_all_star", score_col="final_score"):
    df_sorted = df.sort_values(score_col, ascending=False)
    top_k = df_sorted.head(k)

    tp = int(top_k[label_col].sum())
    precision = tp / k if k > 0 else 0.0

    total_positives = int(df[label_col].sum())
    recall = tp / total_positives if total_positives > 0 else 0.0

    return precision, recall

def compute_final_score(df, alpha, beta):
    return (
        alpha * df["p_ensemble"]
        + (1.0 - alpha) * (beta * df["espn_score"] + (1.0 - beta) * df["popularity_score"]) )


In [ ]:
all_stars_by_season = {
    "2019-2020": all_stars_19_20,
    "2020-2021": all_stars_20_21,
    "2021-2022": all_stars_21_22,
    "2022-2023": all_stars_22_23,
}

for season_label, all_star_list in all_stars_by_season.items():
    mask = df_train_final["season"] == season_label
    if not mask.any():
        continue

    all_star_set = set(all_star_list)
    df_train_final.loc[mask, "is_all_star"] = (
        df_train_final.loc[mask, "Player"].isin(all_star_set).astype(int)
    )

def evaluate_alpha_beta(alpha, beta, df_train_final):
    df = df_train_final.copy()
    df["final_score"] = compute_final_score(df, alpha, beta)

    precisions = []
    recalls = []

    for season_label, season_df in df.groupby("season"):
        k = int(season_df["is_all_star"].sum())
        if k == 0:
            continue

        precision, recall = precision_recall_at_k(
            season_df,
            k,
            label_col="is_all_star",
            score_col="final_score"
        )
        precisions.append(precision)
        recalls.append(recall)

    if not precisions:
        return 0.0, 0.0

    avg_precision = float(np.mean(precisions))
    avg_recall = float(np.mean(recalls))
    return avg_precision, avg_recall



> Grid Search to Find the Best Alpha and Beta that Maximizes Precision / Recall



In [ ]:
alphas = np.linspace(0.0, 1.0, 101)
betas  = np.linspace(0.0, 1.0, 101)

best_alpha, best_beta = None, None
best_precision = -1.0
best_recall_for_best = -1.0

for a in alphas:
    for b in betas:
        avg_precision, avg_recall = evaluate_alpha_beta(a, b, df_train_final)

        # prioritize precision, break ties by recall
        if (avg_precision > best_precision) or (
            np.isclose(avg_precision, best_precision) and avg_recall > best_recall_for_best
        ):
            best_precision = avg_precision
            best_recall_for_best = avg_recall
            best_alpha, best_beta = a, b

print(f"Best alpha: {best_alpha:.5f}")
print(f"Best beta:  {best_beta:.5f}")
print(f"Best avg precision@k: {best_precision:.3f}")
print(f"Best avg recall@k:    {best_recall_for_best:.3f}")

Best alpha: 0.78000
Best beta:  0.00000
Best avg precision@k: 0.979
Best avg recall@k:    0.979




> Test Precision / Recall



In [ ]:
# check best_alpha & best_beta
print(f"Best alpha: {best_alpha:.3f}")
print(f"Best beta:  {best_beta:.3f}")

# add is_all_star column
all_star_set_23_24 = set(all_stars_23_24)
df_test_final["is_all_star"] = df_test_final["Player"].isin(all_star_set_23_24).astype(int)

df_test_final["final_score"] = compute_final_score(df_test_final, best_alpha, best_beta)
precision_25, recall_25 = precision_recall_at_k(df_test_final, 26)
print(f"Precision@26: {precision_25:.2f}")
print(f"Recall@26:    {recall_25:.2f}")

Best alpha: 0.780
Best beta:  0.000
Precision@26: 0.73
Recall@26:    0.79


# Prediction for Season 25-26

In [ ]:
# from previous best alpha / best beta
best_alpha = 0.780
best_beta = 0.000

def compute_final_score(df, alpha, beta):
    return (alpha * df["p_ensemble"] + (1.0 - alpha) * (df["popularity_score"]) )

model_1_future = pd.read_csv("2026_All_Star_Predictions.csv")
model_2_future_espn = pd.read_csv("espn_sentiment_score_25_26.csv")
model_2_future_popularity = pd.read_csv("google_trends_pred_25_26.csv")

model_2_future_popularity = model_2_future_popularity.rename(columns={"player": "Player"})

df_future_25_26 = (
    model_1_future[["Player", "p_ensemble"]]
    .merge(
        model_2_future_popularity[["Player", "final_score"]]
        .rename(columns={"final_score": "popularity_score"}),
        on="Player",
        how="inner"
    )
)

df_future_25_26["final_score"] = compute_final_score(df_future_25_26, best_alpha, best_beta)
df_future_25_26.head(24)


,Player,p_ensemble,popularity_score,final_score
0,Nikola Jokić,0.877654,0.094545,0.705370
1,Luka Dončić,0.867920,0.123949,0.704246
2,Victor Wembanyama,0.833824,0.195543,0.693402
3,James Harden,0.777589,0.172348,0.644436
4,Shai Gilgeous-Alexander,0.493330,0.130216,0.413445
5,Giannis Antetokounmpo,0.481262,0.141585,0.406533
6,Tyrese Maxey,0.474808,0.128534,0.398627
7,Donovan Mitchell,0.457585,0.124350,0.384274
8,Austin Reaves,0.437813,0.174496,0.379883
9,Stephen Curry,0.395345,0.218595,0.356460
